# DJF ITCZ wind-divergence climatology

This notebook only plots the 1981-2020 DJF climatology Min_Div ITCZ line from 850 hPa u/v winds.
The goal is to check whether the wind-divergence ITCZ line looks patchy.

- DJF labeling follows Dec(year-1) + Jan(year) + Feb(year) = DJF(year).
- ITCZ is extracted as the latitude of minimum divergence within 20S-20N for each longitude.
- No precipitation is used.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import metpy.calc as mpcalc
from metpy.units import units

MPLCONFIGDIR = Path('/tmp/matplotlib')
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPLCONFIGDIR))

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']

FIGURE_DIR = Path('.')
U_V_PATH = Path('/Users/rizzie/TugasAkhir/data_processing/external/ClimateData/era5-monthly/u_v_global850.nc')

GLOBAL_EXTENT = [0, 360, -25, 25]
GLOBAL_LON_TICKS = np.arange(0, 360, 30)
GLOBAL_LAT_TICKS = np.arange(-25, 26, 5)
MC_EXTENT = [90, 155, -20, 20]
MC_LON_TICKS = np.arange(90, 156, 10)
MC_LAT_TICKS = np.arange(-20, 21, 5)


In [ ]:
# Load the ERA5 850 hPa winds. The file contains a singleton pressure_level, so drop it safely.
ds = xr.open_dataset(U_V_PATH)[['u', 'v']]
if 'pressure_level' in ds.dims:
    if ds.sizes['pressure_level'] == 1:
        ds = ds.isel(pressure_level=0, drop=True)
    else:
        ds = ds.sel(pressure_level=850.0)
        if 'pressure_level' in ds.dims:
            ds = ds.isel(pressure_level=0, drop=True)
ds = ds.rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
ds = ds.assign_coords(lon=(ds.lon % 360)).sortby('lon').sortby('lat')
ds = ds.sel(time=slice('1980-12-01', '2020-02-29'))

u850 = ds['u']
v850 = ds['v']

print('u850 dims:', dict(u850.sizes))
print('v850 dims:', dict(v850.sizes))


In [ ]:
def make_djf_means(da: xr.DataArray, start_year: int = 1981, end_year: int = 2020) -> xr.DataArray:
    djf = da.sel(time=da.time.dt.month.isin([12, 1, 2])).copy()
    time_index = pd.DatetimeIndex(djf['time'].values)
    djf_year_values = np.where(time_index.month == 12, time_index.year + 1, time_index.year).astype(int)
    djf = djf.assign_coords(djf_year=('time', djf_year_values))
    season_counts = pd.Series(djf_year_values).value_counts().sort_index()
    complete_years = season_counts[season_counts == 3].index.to_numpy(dtype=int)
    complete_years = complete_years[(complete_years >= start_year) & (complete_years <= end_year)]
    return djf.groupby('djf_year').mean('time').sel(djf_year=complete_years)

def compute_divergence(u_field: xr.DataArray, v_field: xr.DataArray) -> xr.DataArray:
    lons = np.asarray(u_field['lon'].values, dtype=float)
    lats = np.asarray(u_field['lat'].values, dtype=float)
    dx, dy = mpcalc.lat_lon_grid_deltas(lons, lats)
    div = mpcalc.divergence(
        u_field.values * units('m/s'),
        v_field.values * units('m/s'),
        dx=dx,
        dy=dy,
    )
    return xr.DataArray(
        div.magnitude,
        coords={'lat': u_field['lat'], 'lon': u_field['lon']},
        dims=('lat', 'lon'),
        name='divergence',
    )

# Note: wind-divergence ITCZ is generally more reliable over oceans than continents.
def extract_min_div_line(div_field: xr.DataArray, lat_min: float = -20.0, lat_max: float = 20.0) -> xr.DataArray:
    band = div_field.sel(lat=slice(lat_min, lat_max))
    line = band.idxmin('lat')
    line.name = 'itcz_latitude'
    return line


In [ ]:
# Build the DJF climatology and the Min_Div ITCZ line.
u_djf = make_djf_means(u850).transpose('djf_year', 'lat', 'lon')
v_djf = make_djf_means(v850).transpose('djf_year', 'lat', 'lon')

u_clim = u_djf.mean('djf_year').transpose('lat', 'lon')
v_clim = v_djf.mean('djf_year').transpose('lat', 'lon')
div_clim = compute_divergence(u_clim, v_clim)
itcz_line = extract_min_div_line(div_clim, lat_min=-20.0, lat_max=20.0)
itcz_zonal = float(itcz_line.mean('lon', skipna=True))

print('DJF years used:', int(u_djf.sizes['djf_year']))
print('ITCZ line points:', int(itcz_line.count()))
print('Zonal ITCZ latitude:', itcz_zonal)


In [ ]:
def plot_itcz_line(line: xr.DataArray, title: str, extent, xticks, yticks, save_path: Path, figsize=(13.5, 6.5)) -> None:
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))
    ax.plot(
        line['lon'].values,
        line.values,
        color='black',
        linewidth=2.0,
        transform=ccrs.PlateCarree(),
        zorder=3,
    )
    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=2)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=2)
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.legend([Line2D([0], [0], color='black', lw=2.0)], ['Climatology Min_Div ITCZ'], loc='lower left', frameon=True)
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved figure: {save_path}')


In [ ]:
plot_itcz_line(
    itcz_line,
    title='DJF 850 hPa wind-divergence ITCZ climatology (1981-2020) - Global',
    extent=GLOBAL_EXTENT,
    xticks=GLOBAL_LON_TICKS,
    yticks=GLOBAL_LAT_TICKS,
    save_path=FIGURE_DIR / 'itcz_wind_clim_global.png',
    figsize=(13.5, 6.5),
)


In [ ]:
plot_itcz_line(
    itcz_line,
    title='DJF 850 hPa wind-divergence ITCZ climatology (1981-2020) - Maritime Continent',
    extent=MC_EXTENT,
    xticks=MC_LON_TICKS,
    yticks=MC_LAT_TICKS,
    save_path=FIGURE_DIR / 'itcz_wind_clim_mc_90E_155E.png',
    figsize=(12.5, 6.5),
)
